In [0]:
# Define job inputs
dbutils.widgets.text("catalog_name", "main", "UC catalog name")
dbutils.widgets.text("schema_name", "default", "UC schema name")
dbutils.widgets.text("num_training_rows", "100", "Rows of data")
dbutils.widgets.text("num_training_columns", "1000", "Number of features")
dbutils.widgets.text("num_labels", "2", "Number labels in target column")
dbutils.widgets.text("registered_model_name", "dummy_data_classifier", "Registered model name")
dbutils.widgets.text("serving_endpoint_name", "dummy_classifier", "Serving endpoint name")

# Get parameter values (will override widget defaults if run by job)
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
num_training_rows = int(dbutils.widgets.get("num_training_rows"))
num_training_columns = int(dbutils.widgets.get("num_training_columns"))
num_labels = int(dbutils.widgets.get("num_labels"))
registered_model_name = dbutils.widgets.get("registered_model_name")
serving_endpoint_name = dbutils.widgets.get("serving_endpoint_name")

# Generated values
table = f"synthetic_data_{num_training_rows}_rows_{num_training_columns}_columns_{num_labels}_labels"
full_model_name = f"{catalog_name}.{schema_name}.{registered_model_name}"
label_column_name="target"

In [0]:
import mlflow
from mlflow import MlflowClient 
from sklearn.metrics import classification_report

client = MlflowClient()
# Champion // Challenger bake-off

# Edit to point to an unseen test dataset. For demo purposes, we're just grabbing 20% of our dataset. 
unseen_test_df = spark.read.table(f'{catalog_name}.{schema_name}.{table}').sample(fraction=0.2, seed=42).toPandas()
y_test = unseen_test_df.pop(label_column_name)

try:
  champion_model_uri = f"models:/{full_model_name}@champion"
  champion_loaded_model = mlflow.sklearn.load_model(champion_model_uri)
  print('champion model loaded')

  challenger_model_uri = f"models:/{full_model_name}@challenger"
  challenger_loaded_model = mlflow.sklearn.load_model(challenger_model_uri)
  print('challenger model loaded')

  champ_pred = champion_loaded_model.predict(unseen_test_df)
  challenger_pred = challenger_loaded_model.predict(unseen_test_df)

  champ_report = classification_report(y_test, champ_pred, output_dict=True)
  chall_report = classification_report(y_test, challenger_pred, output_dict=True)
except:
  print('bakeoff failed')


In [0]:
champ_f1 = champ_report['weighted avg']['f1-score']
chall_f1 = chall_report['weighted avg']['f1-score']

# Default redeployment to FALSE
redeploy=False

if chall_f1 >= champ_f1:
  print('Challenger model is better. Promoting...')
  challenger_model = client.get_model_version_by_alias(full_model_name, "Challenger")
  client.set_registered_model_alias(
    name=full_model_name,
    alias="Champion",
    version=challenger_model.version
  )
  deploy_version = challenger_model.version
  redeploy=True
else:
  print('Champion model is better. No promotion needed.')


In [0]:
from databricks.sdk.service.serving import EndpointTag
from databricks.sdk.service.serving import EndpointCoreConfigInput
from databricks.sdk import WorkspaceClient

# Create workspace client for the Databricks Python SDK
w = WorkspaceClient()

if redeploy:
  endpoint_config_dict = {
      "served_models": [
          # Add models to be served to this list
          {
              "model_name": full_model_name,
              "model_version": deploy_version,
              "scale_to_zero_enabled": True,
              "workload_size": "Small",
          },
      ],
      "traffic_config": {
          "routes": [
              # Add versions of the model to be served to this list
              # Make sure that traffic_percentage adds up to 100 over all served models
              # Naming convention for served_model_name: <registered_model_name>-<model_version>
              {"served_model_name": f"{registered_model_name}-{deploy_version}", "traffic_percentage": 100},
          ]
      }
  }

  endpoint_config = EndpointCoreConfigInput.from_dict(endpoint_config_dict)

  try:
    # Create and do not wait. Check the readiness of the endpoint in the next cell.
    w.serving_endpoints.create(
      name=serving_endpoint_name,
      config=endpoint_config)
    
    print(f"Creating endpoint {serving_endpoint_name} with models {full_model_name} version {deploy_version}")

  except Exception as e:
    if f"Endpoint with name '{serving_endpoint_name}' already exists" in e.args[0]:
      print(f"Endpoint with name {serving_endpoint_name} already exists, updating it with model {full_model_name}-{deploy_version} ({str(e)})")

      w.serving_endpoints.update_config(
        name=serving_endpoint_name,
        served_models=endpoint_config.served_models,
        traffic_config=endpoint_config.traffic_config
      )
    else:
      raise(e)